# 03 · Índices de riesgo y escala normativa

**Objetivo.** Caracterizar el riesgo por tres vías complementarias:

1. **PLI** (índice de carga de contaminación): media geométrica de los
   factores de contaminación de 8 metales.
2. **RI** (riesgo ecológico potencial de Hakanson): suma ponderada por
   toxicidad.
3. **Nivel de riesgo normativo** (0-3): cuántos metales superan ISQG y PEL
   de las guías CCME.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
FIGURES = ROOT / "reports" / "figures"

In [ ]:
riesgo = pd.read_csv(PROCESSED / "dataset_riesgo.csv", encoding="utf-8-sig")
riesgo[["PLI", "RI", "nivel_pli", "nivel_ri", "nivel_riesgo",
        "n_metales_sobre_isqg", "n_metales_sobre_pel"]].describe(include="all").T

## 1. Distribución del PLI

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist(riesgo["PLI"].dropna(), bins=30, color="#2c7bb6", edgecolor="white")
axes[0].axvline(1, color="#333", ls="--", lw=1, label="PLI = 1")
axes[0].set_xlabel("PLI")
axes[0].set_ylabel("Muestras")
axes[0].legend()

conteo = riesgo["nivel_pli"].value_counts().reindex(["Bajo", "Medio", "Alto", "Muy alto"])
axes[1].bar(conteo.index, conteo.values, color=["#2c7bb6", "#ffffbf", "#fdae61", "#d7191c"],
            edgecolor="k", linewidth=0.4)
axes[1].set_title("Nivel según PLI")
plt.tight_layout()
plt.show()

print("PLI mediano:", round(riesgo["PLI"].median(), 2))
print("% PLI > 1:", round(100 * riesgo["PLI"].gt(1).mean(), 1))

**Lectura.** El PLI mediano es 2.44 y ~80 % de las muestras superan 1. Un
umbral tan laxo no discrimina, por eso el nivel de riesgo principal se ancla
en las guías CCME y no en el PLI.

## 2. Escala normativa CCME (ISQG / PEL)

In [ ]:
orden = ["Bajo", "Medio", "Alto", "Muy alto", "Sin dato"]
conteo = riesgo["nivel_riesgo"].value_counts().reindex(orden).fillna(0).astype(int)
tabla = pd.DataFrame({"muestras": conteo, "porcentaje": (100 * conteo / len(riesgo)).round(1)})
tabla

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
colores = ["#2c7bb6", "#ffffbf", "#fdae61", "#d7191c", "#bdbdbd"]
b = ax.bar(conteo.index, conteo.values, color=colores, edgecolor="k", linewidth=0.4)
for barra, v in zip(b, conteo.values):
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 2,
            f"{v}\n({100*v/len(riesgo):.0f} %)", ha="center", fontsize=8)
ax.set_ylabel("Muestras")
ax.set_title("Nivel de riesgo normativo (CCME ISQG/PEL)")
ax.set_ylim(0, conteo.max() * 1.3)
plt.tight_layout()
plt.show()

**Lectura.** El nivel *Alto* (≥1 metal sobre PEL) concentra el mayor número de
muestras. El criterio es acumulativo: *Muy alto* exige 3 o más metales sobre
PEL, lo que separa los focos realmente críticos.

## 3. ¿Qué metal manda en cada muestra?

In [ ]:
riesgo["metal_dominante"].value_counts().head(10).to_frame("muestras")

In [ ]:
excedencias = {}
for m in cfg.METALES_INDICE:
    col = f"ratio_pel_{m}"
    if col in riesgo.columns:
        excedencias[m] = 100 * riesgo[col].gt(1).mean()
pd.Series(excedencias, name="% sobre PEL").sort_values(ascending=False).round(1).to_frame()

## 4. Riesgo ecológico (RI de Hakanson)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(np.log1p(riesgo["RI"].dropna()), bins=30, color="#d7191c", edgecolor="white")
ax.set_xlabel("log1p(RI)")
ax.set_ylabel("Muestras")
ax.set_title("Riesgo ecológico potencial (RI)")
plt.tight_layout()
plt.show()

riesgo.groupby("nivel_ri")["RI"].agg(["count", "median"]).round(1)

## 5. Focos por distrito

In [ ]:
tabla = (
    riesgo.groupby("DISTRITO")
    .agg(muestras=("PLI", "size"),
         PLI_mediano=("PLI", "median"),
         pct_alto=("nivel_riesgo", lambda s: 100 * s.isin(["Alto", "Muy alto"]).mean()))
    .sort_values("pct_alto", ascending=False)
    .round(1)
)
tabla

**Lectura.** El porcentaje de muestras en riesgo alto o muy alto es el
indicador que alimenta el ranking de puntos críticos y el mapa del tablero.